<a href="https://colab.research.google.com/github/mooch443/dataset-fixer/blob/main/notebooks/02_task_aware_tiling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/></a>

# Task-aware image tiling

This tutorial demonstrates `Dataset.tile()` in regular overlapping-grid mode for bounding boxes and randomized coverage mode for POLO point annotations.

> **AI-generation disclosure:** this project and tutorial are largely AI-generated under human direction and review. Independently validate results for your data.

All generated example images and annotations are released under this repository's [MIT License](../LICENSE).

In [ ]:
import os
if not os.path.isdir('/content/dataset-fixer'):
    !git clone -q https://github.com/mooch443/dataset-fixer.git /content/dataset-fixer
%cd /content/dataset-fixer
%pip install -q -e .

In [ ]:
from pathlib import Path
from dataset_fixer import Dataset
from examples.create_example_datasets import create_example_datasets

paths = create_example_datasets('/content/dataset-fixer-examples', seed=42)
detection = Dataset.open(paths['detection'], task='detect')
polo = Dataset.open(paths['polo'], task='polo')
print(detection)
print(polo)

## 1. Regular grid tiling for detection

The grid uses edge-aligned final windows without resizing. Bounding boxes are clipped and retained only when the configured fraction of their original area remains. The preview shows the exact windows before processing.

In [ ]:
import shutil
grid_destination = Path('/content/orchard-grid-tiles')
if grid_destination.exists():
    shutil.rmtree(grid_destination)

grid = detection.tile(
    mode='grid',
    tile_size=320,
    overlap=0.20,
    min_area_ratio=0.10,
    negative_tiles='all',
    destination=grid_destination,
    visualize=True,
)
print(grid)
grid.visualize(split='val', n=8, seed=42, columns=4)

## 2. Randomized coverage tiling for POLO

Coverage mode chooses randomized zoomed crops while tracking how often every point is included. The full fixed-radius source circle must fit inside a positive crop, and background crops may not touch any point circle.

The tutorial uses lower coverage targets than the production defaults so it runs quickly on a Colab CPU. Every override is stored in the output manifest.

In [ ]:
coverage_destination = Path('/content/orchard-coverage-tiles')
if coverage_destination.exists():
    shutil.rmtree(coverage_destination)

coverage = polo.tile(
    mode='coverage',
    tile_size=480,
    destination=coverage_destination,
    target_coverage_per_label=2,
    sparse_coverage_per_label=1,
    max_bg_ratio=0.10,
    fixed_polo_radius_px=18,
    seed=42,
    visualize=True,
)
print(coverage)
print('coverage reports:', coverage.location / 'coverage_summary')

## 3. Inspect coverage and provenance

Each tile records its crop, scale, tile index, parent image, and ultimate original. Coverage CSVs remain available even when raster audits are disabled.

In [ ]:
import pandas as pd
from IPython.display import display

summary = pd.read_csv(coverage.location / 'coverage_summary' / 'label_coverage.csv')
display(summary.head(10))
print('mean achieved coverage:', summary['actual_coverages'].mean())
print('output images:', len(coverage.provenance))
first_record = next(iter(coverage.provenance.values()))
display(first_record)

### Annotation rules by task

- **Detection:** boxes are clipped and area-filtered.
- **Segmentation:** polygons are geometrically intersected with each crop.
- **Pose:** outside keypoints become `(0, 0, 0)` and insufficient instances are removed.
- **POLO grid:** a point is kept only if its complete configured radius circle fits.
- **POLO coverage:** crops are resized to the output resolution and radii are scaled accordingly.